# Native 物理更新的家族稳健性、来源预测与状态对齐（CPU v1）

接在已完成的 Native PhysicalUpdate Confirmation v2 之后。

[实验设置与运行说明](../../../docs/experiment_notes/06_appendix_physical_updates/07b60530a3690474.md)


In [ ]:
# Run once. Every later code cell reports both cell and inner-loop progress.
import re
import threading
import time
from IPython.core.magic import register_cell_magic
from tqdm.auto import tqdm

_SETUP_PROGRESS = tqdm(total=1, desc='Initialize progress display', unit='cell')
_ACTIVE_CELL_PROGRESS = None

def cell_stage(message):
    if _ACTIVE_CELL_PROGRESS is not None:
        _ACTIVE_CELL_PROGRESS['stage'] = str(message)
        elapsed = time.monotonic() - _ACTIVE_CELL_PROGRESS['started']
        _ACTIVE_CELL_PROGRESS['bar'].set_postfix_str(
            f"{message} | elapsed {elapsed:.0f}s", refresh=True
        )

@register_cell_magic
def cell_progress(line, cell):
    global _ACTIVE_CELL_PROGRESS
    specification = line.strip() or 'Cell progress'
    match = re.fullmatch(r'(\d+)/(\d+)\s+(.+)', specification)
    if match:
        current, total, label = int(match.group(1)), int(match.group(2)), match.group(3)
        initial = current - 1
    else:
        current, total, label, initial = 1, 1, specification, 0
    started = time.monotonic()
    bar = tqdm(total=total, initial=initial, desc=label, unit='cell',
               dynamic_ncols=True, leave=True)
    state = {'bar': bar, 'started': started, 'stage': 'starting'}
    _ACTIVE_CELL_PROGRESS = state
    stopped = threading.Event()

    def heartbeat():
        while not stopped.wait(1.0):
            elapsed = time.monotonic() - started
            bar.set_postfix_str(
                f"{state['stage']} | elapsed {elapsed:.0f}s", refresh=True
            )

    worker = threading.Thread(target=heartbeat, daemon=True)
    worker.start()
    succeeded = False
    try:
        cell_stage('running')
        result = get_ipython().run_cell(cell)
        result.raise_error()
        succeeded = True
        return result.result
    finally:
        stopped.set()
        worker.join(timeout=2.0)
        elapsed = time.monotonic() - started
        if succeeded:
            bar.n = current
            bar.set_postfix_str(f"complete | elapsed {elapsed:.0f}s", refresh=True)
        else:
            bar.set_postfix_str(f"failed | elapsed {elapsed:.0f}s", refresh=True)
        bar.close()
        _ACTIVE_CELL_PROGRESS = None

_SETUP_PROGRESS.update(1)
_SETUP_PROGRESS.close()
print('Cell progress helper ready: 13 tracked cells.')

In [ ]:
%%cell_progress 1/13 Prepare CPU dependencies
cell_stage('Prepare CPU dependencies')
import importlib.util, importlib.metadata, subprocess, sys
requirements={'numpy':'numpy>=1.26','pandas':'pandas>=2.2','matplotlib':'matplotlib>=3.8','tqdm':'tqdm>=4.66','IPython':'ipython>=8.0'}
missing=[requirement for module,requirement in requirements.items() if importlib.util.find_spec(module) is None]
if missing:subprocess.check_call([sys.executable,'-m','pip','install','-q',*missing])
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm
from IPython.display import display, Markdown
print('CPU only: no Torch, model weights, training or inference.')
print({name:importlib.metadata.version(name) for name in requirements})


In [ ]:
%%cell_progress 2/13 Locate the original Native review bundle
cell_stage('Locate the original Native review bundle')
from pathlib import Path
SOURCE_ZIP_OVERRIDE=''  # Optional full path to the ORIGINAL Native GPU review_bundle.zip.
OUTPUT_DIR_OVERRIDE=''
ALLOW_SYNTHETIC_FIXTURE=False  # Implementation tests only; keep False for your real result.
try:in_colab=importlib.util.find_spec('google.colab') is not None
except ModuleNotFoundError:in_colab=False
if in_colab:
    from google.colab import drive
    drive.mount('/content/drive')
relative='results/qwen27b_physical_update_answercode_confirmation_v1/native/review_bundle.zip'
roots=[Path('/content/drive/MyDrive/VisualTorque'),Path('/content/drive/MyDrive/VisualTorque/VisualTorque'),Path.cwd(),Path.cwd()/'VisualTorque']
if SOURCE_ZIP_OVERRIDE:SOURCE_ZIP=Path(SOURCE_ZIP_OVERRIDE).expanduser().resolve()
else:
    found=sorted({(root/relative).resolve() for root in roots if (root/relative).is_file()})
    if len(found)!=1:raise FileNotFoundError('Set SOURCE_ZIP_OVERRIDE to the full path of the Native PhysicalUpdate GPU review_bundle.zip. Found: '+str(found))
    SOURCE_ZIP=found[0]
if not SOURCE_ZIP.is_file():raise FileNotFoundError(SOURCE_ZIP)
default=(SOURCE_ZIP.parent.parent.parent if SOURCE_ZIP.parent.name=='native' else SOURCE_ZIP.parent)/'qwen27b_native_physical_update_cpu_audit_v1'
OUTPUT_BASE=Path(OUTPUT_DIR_OVERRIDE).expanduser().resolve() if OUTPUT_DIR_OVERRIDE else default
print('Read-only original GPU bundle:',SOURCE_ZIP)


In [ ]:
%%cell_progress 3/13 Freeze the CPU analysis settings
cell_stage('Freeze the CPU analysis settings')
import hashlib, io, itertools, json, math, platform, zipfile
from collections import Counter
from functools import lru_cache
from pathlib import PurePosixPath
from datetime import datetime, timezone
IMPLEMENTATION='960837ce657951b2b865b28bc1026e6b8d1e2ff76f69a283b60286224e347bc2'
AUDIT_VERSION='native-physical-update-cpu-1.0'
AUDIT_SEED=20260907
AUDIT_BOOTSTRAPS=10000
LABELS=('A','B','C')
SEMANTICS=('left','balanced','right')
KINDS=('physical','physical_half','reverse','random','background','mapping')
PAIR=['family_id','input_key','context','source_mapping_slot']
def require(ok,message):
    if not bool(ok):raise ValueError(message)
def digest(obj):return 'sha256:'+hashlib.sha256(json.dumps(obj,sort_keys=True,separators=(',',':'),ensure_ascii=False,allow_nan=False).encode()).hexdigest()
def bytes_sha(data):return 'sha256:'+hashlib.sha256(data).hexdigest()
def file_sha(path):
    h=hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda:f.read(2**20),b''):h.update(chunk)
    return 'sha256:'+h.hexdigest()
def write_json(path,obj):
    path=Path(path);tmp=path.with_suffix(path.suffix+'.tmp')
    tmp.write_text(json.dumps(obj,ensure_ascii=False,sort_keys=True,allow_nan=False,indent=2)+'\n',encoding='utf-8');tmp.replace(path)
def freeze(path,obj):
    obj=json.loads(json.dumps(obj,ensure_ascii=False,allow_nan=False));payload={'identity':obj,'digest':digest(obj)}
    if path.is_file():require(json.loads(path.read_text(encoding='utf-8'))==payload,'Input/settings changed; use a different OUTPUT_DIR_OVERRIDE: '+str(path))
    else:write_json(path,payload)
    return payload['digest']
def save(frame,name):
    p=TABLE_DIR/(name+'.csv');tmp=p.with_suffix('.csv.tmp');frame.to_csv(tmp,index=False);tmp.replace(p)
def records(frame):return json.loads(frame.to_json(orient='records',double_precision=15))
def show(frame,limit=70):
    with pd.option_context('display.max_rows',limit,'display.max_columns',30,'display.width',180):display(frame.round(6))
def unique(frame,keys,label):
    require(set(keys)<=set(frame),label+': missing keys.')
    require(not frame[keys].isna().any().any() and not frame.duplicated(keys).any(),label+': null/duplicate keys.')
def strict_bool(series):
    def one(v):
        if isinstance(v,(bool,np.bool_)):return bool(v)
        if isinstance(v,str) and v.lower() in ('true','false'):return v.lower()=='true'
        raise ValueError('Invalid boolean '+str(v)+' in '+str(series.name))
    return series.map(one)
def scores(s,prompt):
    require(s['candidate_labels']==prompt['labels']==list(LABELS) and s['candidate_semantics']==prompt['semantics'],'Candidate identity mismatch.')
    vals=np.asarray(s['candidate_logprobs'],float)
    require(vals.shape==(3,) and np.isfinite(vals).all(),'Invalid candidate scores.')
    idx=int(vals.argmax());require(s['predicted_label']==LABELS[idx] and s['predicted_semantic']==prompt['semantics'][idx],'Prediction mismatch.')
    return dict(zip(prompt['semantics'],vals))
def margin(s,target):return float(s[target]-max(v for k,v in s.items() if k!=target))
def tie_top(s):
    v=list(s.values());return sum(x==max(v) for x in v)>1

SOURCE_HASH=file_sha(SOURCE_ZIP)
OUTPUT_ROOT=OUTPUT_BASE/SOURCE_HASH.split(':')[1][:12]
TABLE_DIR=OUTPUT_ROOT/'tables';FIGURE_DIR=OUTPUT_ROOT/'figures'
require(not SOURCE_ZIP.is_relative_to(OUTPUT_ROOT),'Source must remain outside the output directory.')
for p in (OUTPUT_ROOT,TABLE_DIR,FIGURE_DIR):p.mkdir(parents=True,exist_ok=True)
ANALYSIS_SPEC={'version':AUDIT_VERSION,'implementation':IMPLEMENTATION,'source_zip_hash':SOURCE_HASH,'seed':AUDIT_SEED,
    'bootstrap':AUDIT_BOOTSTRAPS,'unit':'physical family; equal weight among supported families',
    'new_confirmatory_tests':False,'selection':'clean donor predictions only; never patch outcomes',
    'source_subsets':['all','prediction_stable','stable_unique_top','stable_wrong','prediction_changed','source_both_correct','competence_satisfied','all_v2_pairs'],
    'trace_layers':[48,49,55],'undefined_alignment':'NA with explicit coverage; no zero imputation'}
freeze(OUTPUT_ROOT/'analysis_lock.json',ANALYSIS_SPEC)
print('Independent CPU output:',OUTPUT_ROOT)


In [ ]:
%%cell_progress 4/13 Verify source hashes and frozen identities
cell_stage('Verify source hashes and frozen identities')
def safe_member(name):
    p=PurePosixPath(name)
    require(not p.is_absolute() and '..' not in p.parts and '\\' not in name and not any(':' in s for s in p.parts),'Invalid ZIP member path.')
def open_bundle(path):
    with zipfile.ZipFile(path) as z:
        infos=[i for i in z.infolist() if not i.is_dir()];names=[i.filename for i in infos]
        require(sum(i.file_size for i in infos)<=512*1024**2,'Uncompressed bundle exceeds 512 MiB.')
        require(len(names)==len(set(names)),'Duplicate ZIP members.')
        for n in names:safe_member(n)
        roots=[n[:-len('model/RUN_COMPLETE.json')] for n in names if n.endswith('model/RUN_COMPLETE.json')]
        roots=[r for r in roots if r+'shared/frozen_protocol.json' in names]
        require(len(roots)==1,'Expected exactly one Native PhysicalUpdate GPU bundle (model/ and shared/).')
        root=roots[0];return {n[len(root):]:z.read(n) for n in names if n.startswith(root)}
BLOB=open_bundle(SOURCE_ZIP)
@lru_cache(maxsize=None)
def read_json(name):
    require(name in BLOB,'Missing source artifact: '+name)
    return json.loads(BLOB[name].decode('utf-8'),parse_constant=lambda v:(_ for _ in ()).throw(ValueError('Nonfinite JSON: '+v)))
def identity(name):
    obj=read_json(name);require(set(('identity','digest'))<=set(obj) and digest(obj['identity'])==obj['digest'],'Frozen identity digest mismatch: '+name)
    return obj['identity']
def read_table(name):return pd.read_csv(io.BytesIO(BLOB['model/analysis/'+name+'.csv']),float_precision='round_trip')
completion=read_json('model/RUN_COMPLETE.json')
require(completion['status']=='complete' and completion['mode']=='native','A completed Native run is required.')
for prefix,key in (('model/','outputs'),('shared/','shared_outputs')):
    for name,value in tqdm(completion[key].items(),desc='Verify '+prefix+'file hashes',unit='file'):
        safe_member(name);require(prefix+name in BLOB,'Missing completed artifact: '+prefix+name)
        require(bytes_sha(BLOB[prefix+name])==value,'Artifact hash mismatch: '+prefix+name)
verified_members={'model/'+n for n in completion['outputs']}|{'shared/'+n for n in completion['shared_outputs']}|{'model/RUN_COMPLETE.json'}
require(set(BLOB)==verified_members,'Unmanifested or missing bundle contents.')
PROTOCOL=identity('shared/frozen_protocol.json');MODE=identity('model/model_identity.json');RUNTIME=identity('model/runtime_lock.json')
RUN_ID=identity('model/run_identity.json')['run_id']
require(PROTOCOL['schema']=='answer-neutral-update-1.0' and PROTOCOL['capture_layers']==[48,49,55] and PROTOCOL['injection_layer']==48,'Unsupported scientific protocol.')
require(MODE['mode']=='native' and completion['protocol_digest']==read_json('shared/frozen_protocol.json')['digest'],'Model/protocol mismatch.')
expected=digest({'protocol':read_json('shared/frozen_protocol.json')['digest'],'runtime':read_json('model/runtime_lock.json')['digest'],'model':MODE})
require(RUN_ID==expected==completion['run_id'],'Run identity mismatch.')
SOURCE_SYNTHETIC=bool(PROTOCOL['synthetic_only'] or MODE['synthetic_only'] or completion['synthetic_only'])
require(not SOURCE_SYNTHETIC or ALLOW_SYNTHETIC_FIXTURE,'Synthetic fixture detected: not research results. Only enable ALLOW_SYNTHETIC_FIXTURE for implementation tests.')
if SOURCE_SYNTHETIC:print('SYNTHETIC ONLY — ALL FOLLOWING NUMBERS ARE IMPLEMENTATION TEST DATA.')
scene_rows=identity('shared/scene_manifest.json');prompts_list=identity('shared/prompt_inventory.json')
require(read_json('shared/prompt_inventory.json')['digest']==PROTOCOL['prompt_digest'],'Prompt digest mismatch.')
SCENES={r['scene_id']:r for r in scene_rows};PROMPTS={p['capture_key']:p for p in prompts_list}
require(len(SCENES)==len(scene_rows) and len(PROMPTS)==len(prompts_list),'Duplicate scenes/prompts.')
FAMILIES=[f['family_id'] for f in PROTOCOL['families']];N_FAMILIES=PROTOCOL['n_families']
require(len(FAMILIES)==len(set(FAMILIES))==N_FAMILIES,'Invalid family inventory.')
PLAN={p['condition_id']:p for p in PROTOCOL['plan']}
require(len(PLAN)==25 and set(p['kind'] for p in PLAN.values())==set(KINDS)|{'zero'},'Condition grid changed.')
LOOKUP={(r['family_id'],r['orientation'],r['role']):r for r in scene_rows}
TARGET_KEYS=[k for k,p in PROMPTS.items() if SCENES[p['scene_id']]['role'] in ('flip_before','stable_before')]
require(len(TARGET_KEYS)==N_FAMILIES*24 and len(PROMPTS)==N_FAMILIES*64 and len(SCENES)==N_FAMILIES*16,'Wrong scene/prompt grid.')
for r in scene_rows:
    require(r['family_id'] in FAMILIES and r['orientation'] in ('anchor','mirror'),'Invalid family/orientation.')
    u,d,v,e=r['values'];truth='left' if u*d>v*e else 'right' if u*d<v*e else 'balanced'
    require(r['outcome']==truth,'Scene truth inconsistent with physics.')
    require(bytes_sha(BLOB['shared/selected_images/'+r['image_name']])==r['image_sha256'],'Scene image fingerprint mismatch.')
for p in prompts_list:
    require(tuple(p['semantics'])==tuple(itertools.permutations(SEMANTICS))[p['mapping_id']],'Mapping index mismatch.')
    require(p['image_sha256']==SCENES[p['scene_id']]['image_sha256'],'Prompt image identity mismatch.')
clean_inventory=identity('model/clean_inventory.json');all_clean_inventory=identity('model/all_clean_inventory.json')
doses=identity('model/frozen_doses.json');DOSES={(r['input_key'],r['condition_id']):r for r in doses}
require(len(DOSES)==len(doses)==len(TARGET_KEYS)*25,'Duplicate/missing frozen doses.')
shard_inventory=identity('model/shard_inventory.json')
require(set(clean_inventory)==set(all_clean_inventory)==set(PROMPTS),'Clean inventory differs from frozen prompts.')
for key,h in all_clean_inventory.items():require(h==clean_inventory[key]==bytes_sha(BLOB['model/clean_records/'+key+'.json']),'Clean record hash mismatch.')
require(set(shard_inventory)=={k+'.json' for k in TARGET_KEYS},'Shard grid mismatch.')
for name,h in shard_inventory.items():require(bytes_sha(BLOB['model/intervention_shards/'+name])==h,'Intervention shard hash mismatch.')
PATCH=identity('model/numerical_audit_patch_v2.json')
require(PATCH['run_id']==RUN_ID and PATCH['protocol_digest']==completion['protocol_digest'],'Numerical patch run mismatch.')
write_json(OUTPUT_ROOT/'source_integrity.json',{'source_zip_hash':SOURCE_HASH,'run_id':RUN_ID,'protocol_digest':completion['protocol_digest'],
    'hashed_artifacts':len(verified_members)-1,'source_synthetic':SOURCE_SYNTHETIC,'tensor_limit':'No clean/patch hidden tensors verified or loaded.'})
print('Identity/hash checks passed:',N_FAMILIES,'families;',len(TARGET_KEYS),'target inputs.')


In [ ]:
%%cell_progress 5/13 Rebuild clean scores and source prediction flags
cell_stage('Rebuild clean scores and source prediction flags')
CLEAN={};BEFORE_AFTER={};behavior=[]
for key,p in PROMPTS.items():
    c=read_json('model/clean_records/'+key+'.json');r=SCENES[p['scene_id']]
    require(c['capture_key']==key and c['run_id']==RUN_ID,'Clean capture metadata mismatch.')
    s=scores(c['score'],p);CLEAN[key]={'record':c,'scene':r,'scores':s,'prompt':p}
    behavior.append({**r,'mapping_id':p['mapping_id'],'capture_key':key,'predicted_semantic':c['score']['predicted_semantic'],
        'predicted_label':c['score']['predicted_label'],'gold_label':LABELS[p['semantics'].index(r['outcome'])],
        'correct':c['score']['predicted_semantic']==r['outcome'],'gold_vs_best_other':margin(s,r['outcome']),
        **{'logprob_'+l:v for l,v in zip(LABELS,c['score']['candidate_logprobs'])}})
behavior=pd.DataFrame(behavior)
def compare_table(rebuilt,saved,keys,fields=None,label='table'):
    unique(rebuilt,keys,label+' rebuilt');unique(saved,keys,label+' saved')
    a=rebuilt.sort_values(keys).reset_index(drop=True);b=saved.sort_values(keys).reset_index(drop=True)
    require(len(a)==len(b),label+': row count mismatch.')
    for k in keys:require(a[k].tolist()==b[k].tolist(),label+': key mismatch '+k)
    for k in (fields if fields is not None else [c for c in a if c not in keys]):
        require(k in b,label+': missing saved field '+k)
        if pd.api.types.is_bool_dtype(a[k]):
            require(a[k].tolist()==strict_bool(b[k]).tolist(),label+': boolean mismatch '+k)
        elif pd.api.types.is_numeric_dtype(a[k]):
            require(np.allclose(a[k].to_numpy(float),pd.to_numeric(b[k]).to_numpy(float),atol=1e-10,rtol=1e-9,equal_nan=True),label+': numeric mismatch '+k)
        else:
            aa=a[k].where(a[k].notna(),'<NULL>').astype(str);bb=b[k].where(b[k].notna(),'<NULL>').astype(str)
            require(aa.tolist()==bb.tolist(),label+': metadata mismatch '+k)
    return True
compare_table(behavior,read_table('behavior_records'),['capture_key'],
    ['family_id','scene_id','orientation','role','mapping_id','predicted_semantic','predicted_label','gold_label','correct','gold_vs_best_other','logprob_A','logprob_B','logprob_C'],label='Clean behavior')
donor_rows=[]
for key in TARGET_KEYS:
    for p in PLAN.values():
        dose=DOSES[(key,p['condition_id'])]
        if p['kind']=='zero':
            require(not dose['source_capture_keys'] and dose['source_semantic'] is None,'Zero condition has a donor.');continue
        a,b,other,mapped=dose['source_capture_keys'];ca,cb=CLEAN[a],CLEAN[b]
        ra,rb=ca['scene'],cb['scene'];target=CLEAN[key]['scene']
        require(ra['family_id']==rb['family_id']==target['family_id'] and ra['orientation']==rb['orientation']==target['orientation'],'Cross-family donor.')
        require(ra['role']==f"source{p['context']}_before" and rb['role']==f"source{p['context']}_after",'Wrong donor endpoints.')
        require(ra['outcome']==rb['outcome']==dose['source_semantic'],'Source ground-truth answer changed.')
        require(ca['prompt']['mapping_id']==cb['prompt']['mapping_id']==dose['source_mapping'] and ca['prompt']['prompt']==cb['prompt']['prompt'],'Source prompt/map changed.')
        require(dose['source_label']==LABELS[ca['prompt']['semantics'].index(ra['outcome'])],'Source gold label changed.')
        require(CLEAN[other]['scene']['role']==f"source{1-p['context']}_before" and CLEAN[other]['prompt']['mapping_id']==dose['source_mapping'],'Background control identity changed.')
        require(CLEAN[mapped]['scene']['scene_id']==ra['scene_id'] and CLEAN[mapped]['prompt']['mapping_id']!=dose['source_mapping'],'Mapping control identity changed.')
        pred_a=ca['record']['score']['predicted_semantic'];pred_b=cb['record']['score']['predicted_semantic']
        stable=pred_a==pred_b;both=pred_a==pred_b==ra['outcome'];tied=tie_top(ca['scores']) or tie_top(cb['scores'])
        item={'family_id':ra['family_id'],'orientation':ra['orientation'],'context':p['context'],'source_mapping_slot':p['source_mapping_slot'],
            'source_mapping':dose['source_mapping'],'source_before_key':a,'source_after_key':b,'source_semantic':ra['outcome'],
            'source_prediction_before':pred_a,'source_prediction_after':pred_b,'prediction_stable':stable,'source_top_tie':tied,
            'stable_unique_top':stable and not tied,'stable_wrong':stable and not both,'prediction_changed':not stable,
            'source_both_correct':both,'source_gold_margin_change':margin(cb['scores'],ra['outcome'])-margin(ca['scores'],ra['outcome'])}
        donor_rows.append(item)
donors=pd.DataFrame(donor_rows).drop_duplicates()
DONOR_KEY=['family_id','orientation','context','source_mapping_slot']
unique(donors,DONOR_KEY,'Donor flags');require(len(donors)==N_FAMILIES*8,'Donor coverage mismatch.')
DONORS={tuple(r[k] for k in DONOR_KEY):r for r in donors.to_dict('records')}
save(behavior.drop(columns=['values']),'reconstructed_clean_behavior');save(donors,'donor_prediction_flags')
print('Clean scores reconstructed; donor prediction flags:',len(donors),'paired endpoints.')
show(donors.groupby(['prediction_stable','source_both_correct','source_top_tie']).agg(donor_pairs=('family_id','size'),families=('family_id','nunique')).reset_index())


In [ ]:
%%cell_progress 6/13 Reconstruct intervention scores and saved state traces
cell_stage('Reconstruct intervention scores and saved state traces')
def audit_level(r):
    actual=float(r['edit']['actual_added_norm']);expected=float(r['dose']['expected_actual_added_norm'])
    require(math.isfinite(actual) and actual>=0 and abs(actual-expected)<1e-5,'Saved injection norm gate failed.')
    a=r.get('injection_audit')
    if a is None:
        require('actual_added_norm_device_fp32' not in r['edit'],'Missing v2 audit metadata.');return 'legacy_v1_scalar_only'
    require(a['schema']==PATCH['schema'] and a['patch_code_digest']==PATCH['patch_code_digest'],'Unknown v2 audit identity.')
    require(all(a[k] is True for k in ('passed','prestate_equal','hook_addition_equal','effective_delta_equal','frozen_cpu_norm_equal')),'Saved vector audit failed.')
    for x,y in (('cached_before_sha256','live_before_sha256'),('actual_after_sha256','expected_after_sha256'),('actual_delta_sha256','expected_delta_sha256')):
        require(a[x]==a[y],'Saved vector fingerprint mismatch.')
    require(all(a[k]==0 for k in ('prestate_max_abs_error','hook_max_abs_error','delta_max_abs_error')),'Saved nonzero vector error.')
    require(a['actual_norm_cpu_fp32']==actual and a['expected_norm_cpu_fp32']==actual and a['frozen_expected_norm']==expected,'Saved CPU norm mismatch.')
    require(a['actual_norm_cpu_fp64']==a['expected_norm_cpu_fp64'] and math.isfinite(a['actual_norm_cpu_fp64']),'Saved FP64 norm mismatch.')
    device=float(r['edit']['actual_added_norm_device_fp32'])
    require(math.isfinite(device) and a['device_norm_fp32']==device and a['device_vs_frozen_abs_error']==abs(device-expected),'Saved device diagnostic mismatch.')
    require(a['legacy_scalar_gate_would_pass']==(abs(device-expected)<1e-5),'Old gate diagnostic mismatch.')
    return 'v2_exact_vectors'

patch_rows=[];trace_rows=[];false_old_gate=0
for key in tqdm(TARGET_KEYS,desc='Rebuild scores and validate raw condition records',unit='input'):
    own=CLEAN[key];row=own['scene'];prompt=own['prompt'];before=own['scores'];mid=prompt['mapping_id']
    ar=LOOKUP[(row['family_id'],row['orientation'],row['role'].replace('_before','_after'))]
    after_key=ar['scene_id']+f'__m{mid}';cf=CLEAN[after_key];natural=cf['scores']
    changed=row['changed_side'];opposite='right' if changed=='left' else 'left';gold=ar['outcome'];base_gold=row['outcome']
    sh=read_json('model/intervention_shards/'+key+'.json')
    require(sh['identity']=={'run_id':RUN_ID,'all_clean_digest':read_json('model/all_clean_inventory.json')['digest'],
        'dose_digest':read_json('model/frozen_doses.json')['digest'],'input_key':key},'Shard identity mismatch.')
    require(len(sh['records'])==25 and {r['condition']['condition_id'] for r in sh['records']}==set(PLAN),'Incomplete condition grid.')
    for r in sh['records']:
        require(digest({k:v for k,v in r.items() if k!='record_digest'})==r['record_digest'],'Condition record checksum failed.')
        p=r['condition'];cid=p['condition_id'];dose=r['dose']
        require(p==PLAN[cid] and dose==DOSES[(key,cid)],'Condition/dose differs from frozen plan.')
        s=r['score'];ss=scores(s,prompt);quality=audit_level(r)
        false_old_gate+=int(quality=='v2_exact_vectors' and not r['injection_audit']['legacy_scalar_gate_would_pass'])
        source=dose['source_semantic'];source_label=dose['source_label']
        base={'family_id':row['family_id'],'scene_id':row['scene_id'],'orientation':row['orientation'],
            'receiver_context':row['role'].split('_')[0],'receiver_mapping':mid,'input_key':key,**p,
            'base_semantic':base_gold,'cf_semantic':gold,'cf_label':LABELS[prompt['semantics'].index(gold)],
            'predicted_semantic':s['predicted_semantic'],'predicted_label':s['predicted_label'],
            'clean_correct':own['record']['score']['predicted_semantic']==base_gold,
            'natural_cf_correct':cf['record']['score']['predicted_semantic']==gold,
            'clean_cf_success':own['record']['score']['predicted_semantic']==gold,'cf_success':s['predicted_semantic']==gold,
            'margin_shift':float((ss[changed]-ss[opposite])-(before[changed]-before[opposite])),
            'natural_margin_shift':float((natural[changed]-natural[opposite])-(before[changed]-before[opposite])),
            'cf_vs_best_other':margin(ss,gold),'clean_cf_vs_best_other':margin(before,gold),
            'cf_best_other_gain':margin(ss,gold)-margin(before,gold),
            'source_semantic':source,'source_label':source_label,
            'source_semantic_conflict':source is not None and source!=gold,
            'source_label_conflict':source_label is not None and source_label!=LABELS[prompt['semantics'].index(gold)],
            'source_semantic_copy':s['predicted_semantic']==source,'source_label_copy':s['predicted_label']==source_label,
            'source_semantic_copy_gain':float(s['predicted_semantic']==source)-float(own['record']['score']['predicted_semantic']==source) if source is not None else np.nan,
            'source_label_copy_gain':float(s['predicted_label']==source_label)-float(own['record']['score']['predicted_label']==source_label) if source_label is not None else np.nan,
            'source_endpoints_correct':False,'relative_payload_norm':dose['relative_norm'],'injection_audit_level':quality,**r['edit'],
            **{'logprob_'+l:v for l,v in zip(LABELS,s['candidate_logprobs'])}}
        if p['kind']!='zero':
            donor=DONORS[(row['family_id'],row['orientation'],p['context'],p['source_mapping_slot'])]
            base.update(donor);base['source_endpoints_correct']=donor['source_both_correct']
        base['competence_satisfied']=base['source_endpoints_correct'] and base['clean_correct'] and base['natural_cf_correct']
        base['cf_accuracy_gain']=int(base['cf_success'])-int(base['clean_cf_success'])
        patch_rows.append(base)
        require(set(r['traces'])=={'48','49','55'},'Missing recorded layer traces.')
        for layer,t in r['traces'].items():
            trace_rows.append({'family_id':row['family_id'],'input_key':key,'condition_id':cid,'kind':p['kind'],
                'receiver_context':base['receiver_context'],'orientation':row['orientation'],'receiver_mapping':mid,'layer':int(layer),**t})
frame=pd.DataFrame(patch_rows);traces=pd.DataFrame(trace_rows)
unique(frame,['input_key','condition_id'],'Interventions');unique(traces,['input_key','condition_id','layer'],'Traces')
require(completion['counts']=={'families':N_FAMILIES,'clean_records':len(CLEAN),'interventions':len(frame),'traces':len(traces)},'Completion counts mismatch.')
saved=read_table('intervention_records')
fields=[c for c in saved if c in frame]
require(set(saved)<=set(frame),'Saved intervention table has unexpected fields.')
compare_table(frame,saved,['input_key','condition_id'],fields,label='Raw-score reconstruction')
compare_table(traces,read_table('state_response_traces'),['input_key','condition_id','layer'],label='Raw trace reconstruction')
zero=frame[frame.kind=='zero']
require(zero.margin_shift.abs().max()<=.15 and zero.actual_added_norm.eq(0).all(),'Zero control failed.')
quality_counts={str(k):int(v) for k,v in frame.injection_audit_level.value_counts().items()}
prior_audit=read_json('model/analysis/injection_audit_summary.json')
require(prior_audit['counts']==quality_counts and prior_audit['new_records_that_would_fail_old_scalar_gate']==false_old_gate,'Audit coverage mismatch.')
save(frame,'reconstructed_interventions');save(traces,'reconstructed_state_traces')
write_json(OUTPUT_ROOT/'reconstruction_checks.json',{'clean_scores':len(behavior),'intervention_records':len(frame),'state_traces':len(traces),
    'raw_scores_match_saved_tables':True,'recorded_audit_coverage':quality_counts,'old_scalar_gate_false_failures':false_old_gate,
    'scope':'Saved vector audit summaries checked; no hidden tensors available for new vector verification.'})
print('Full score reconstruction passed:',len(frame),'interventions;',len(traces),'traces.',quality_counts)


In [ ]:
%%cell_progress 7/13 Reproduce the original four tests and behavior effects
cell_stage('Reproduce the original four tests and behavior effects')
def original_estimate(values,key,test=False):
    values=pd.Series(values,dtype=float).dropna().sort_index();require(values.index.is_unique,'One value per family required.')
    if len(values)<2:return {'n_families':len(values),'estimate':float(values.mean()) if len(values) else None,'ci_low':None,'ci_high':None,'p_two_sided':None}
    x=values.to_numpy();require(np.isfinite(x).all(),'Nonfinite original family effects.')
    h=digest([PROTOCOL['seed'],['stats',key]]).split(':')[1]
    rng=np.random.default_rng(int(h[:15],16));means=[]
    for start in range(0,PROTOCOL['bootstraps'],256):
        count=min(256,PROTOCOL['bootstraps']-start);means.extend(x[rng.integers(len(x),size=(count,len(x)))].mean(1))
    out={'n_families':len(x),'estimate':float(x.mean()),'ci_low':float(np.quantile(means,.025)),'ci_high':float(np.quantile(means,.975))}
    if test:
        exceed=0
        for start in range(0,PROTOCOL['signflips'],1024):
            count=min(1024,PROTOCOL['signflips']-start);signs=rng.integers(0,2,size=(count,len(x)))*2-1
            exceed+=int((np.abs((x*signs).mean(1))>=abs(x.mean())-1e-12).sum())
        out['p_two_sided']=(exceed+1)/(PROTOCOL['signflips']+1)
    return out
def holm(p):
    result=np.zeros(len(p));running=0.
    for rank,i in enumerate(np.argsort(p)):
        running=max(running,(len(p)-rank)*p[i]);result[i]=min(1.,running)
    return result
def estimate(values,key):
    values=pd.Series(values,dtype=float).dropna().sort_index();require(values.index.is_unique,'One value per family required.')
    x=values.to_numpy();require(np.isfinite(x).all(),'Nonfinite family values.')
    if len(x)<2:return {'n_families':len(x),'estimate':float(x.mean()) if len(x) else None,'ci_low':None,'ci_high':None}
    rng=np.random.default_rng(int(digest([AUDIT_SEED,key]).split(':')[1][:15],16));means=[]
    for start in range(0,AUDIT_BOOTSTRAPS,256):
        count=min(256,AUDIT_BOOTSTRAPS-start);means.extend(x[rng.integers(len(x),size=(count,len(x)))].mean(1))
    return {'n_families':len(x),'estimate':float(x.mean()),'ci_low':float(np.quantile(means,.025)),'ci_high':float(np.quantile(means,.975))}
def pivot_pairs(sub,metric='margin_shift'):
    wide=sub[sub.kind!='zero'].pivot(index=PAIR,columns='kind',values=metric)
    require(set(wide)==set(KINDS) and not wide.isna().any().any(),'Incomplete paired controls: '+metric)
    return wide
def contrasts(w):
    return pd.DataFrame({'physical_vs_clean':w.physical,'physical_vs_random':w.physical-w.random,
        'physical_vs_background':w.physical-w.background,'physical_vs_mapping':w.physical-w.mapping})
flip=frame[(frame.receiver_context=='flip')&(frame.kind!='zero')]
w=pivot_pairs(flip);primary=[];original_family=[]
for name,values in contrasts(w).items():
    fam=values.groupby('family_id').mean();require(len(fam)==N_FAMILIES,'Original primary support changed.')
    primary.append({'test':name,**original_estimate(fam,name,True)})
    original_family.extend({'test':name,'family_id':k,'value':float(v)} for k,v in fam.items())
primary=pd.DataFrame(primary);primary['p_holm']=holm(primary.p_two_sided.to_numpy());primary['passed']=(primary.estimate>0)&(primary.p_holm<.05)
compare_table(primary,read_table('primary_four_tests'),['test'],label='Original four primary tests')
compare_table(pd.DataFrame(original_family),read_table('primary_family_contrasts'),['test','family_id'],label='Original primary family values')
physical=frame[frame.kind=='physical'].copy();behavior_effects=[]
for context,g in physical.groupby('receiver_context'):
    fam=g.cf_accuracy_gain.groupby(g.family_id).mean()
    behavior_effects.append({'context':context,'metric':'cf_accuracy_gain_over_clean',**original_estimate(fam,['accuracy_gain',context])})
behavior_effects=pd.DataFrame(behavior_effects)
compare_table(behavior_effects,read_table('behavior_effects'),['context','metric'],label='Original accuracy effects')
stable=behavior_effects.set_index('context').loc['stable'];fg=behavior_effects.set_index('context').loc['flip']
decision={'primary_four_pass':bool(primary.passed.all()),'stable_retention_gate':bool(stable.ci_low>PROTOCOL['stable_retention_ci_lower_bound']),
    'flip_accuracy_gain_ci_positive':bool(fg.ci_low>0),'scope':'frozen candidate additive update at L48; not natural necessity or multiplication proof',
    'native_only':True,'synthetic_only':SOURCE_SYNTHETIC}
decision['label']=('candidate_margin_transfer_with_declared_retention' if decision['primary_four_pass'] and decision['stable_retention_gate']
    else 'candidate_margin_effect_with_retention_unresolved' if decision['primary_four_pass'] else 'prespecified_transfer_not_confirmed')
require(decision==read_json('model/decision.json')==completion['decision'],'Original frozen decision mismatch.')
save(primary,'reproduced_primary_four_tests');save(behavior_effects,'reproduced_accuracy_effects')
print('Original four tests, exact resampling, accuracy effects and decision reproduced. No new confirmatory tests.')
show(primary);show(behavior_effects)


In [ ]:
%%cell_progress 8/13 Analyze prediction-stable donor subsets with paired controls
cell_stage('Analyze prediction-stable donor subsets with paired controls')
# Subset membership is identical for physical and all controls in each donor/receiver pair.
META=['receiver_context','orientation','receiver_mapping','source_mapping','prediction_stable','stable_unique_top','stable_wrong',
      'prediction_changed','source_both_correct','source_top_tie','competence_satisfied','source_before_key','source_after_key']
pair_meta=frame[frame.kind=='physical'].set_index(PAIR)[META].copy()
for name in META:
    require(frame[frame.kind!='zero'].groupby(PAIR)[name].nunique(dropna=False).eq(1).all(),'Subset metadata differs across controls: '+name)
pair_meta['all']=True
pair_meta['all_v2_pairs']=frame[frame.kind!='zero'].groupby(PAIR).injection_audit_level.apply(lambda s:s.eq('v2_exact_vectors').all())
SUBSET_NAMES=ANALYSIS_SPEC['source_subsets']
subset_rows=[];subset_families=[];coverage=[]
for context in ('flip','stable'):
    full=frame[(frame.receiver_context==context)&(frame.kind!='zero')]
    wide=pivot_pairs(full);c=contrasts(wide)
    acc=pivot_pairs(full,'cf_accuracy_gain');best=pivot_pairs(full,'cf_best_other_gain')
    c['accuracy_gain_vs_clean']=acc.physical;c['three_candidate_margin_gain']=best.physical
    meta=pair_meta.reindex(c.index)
    for name in SUBSET_NAMES:
        mask=meta[name].fillna(False).astype(bool);part=c.loc[mask]
        idx=part.index.to_frame(index=False)
        coverage.append({'receiver_context':context,'subset':name,'donor_receiver_pairs':len(part),'total_pairs':len(c),
            'distinct_receiver_inputs':idx.input_key.nunique(),'n_families':idx.family_id.nunique(),
            'source_endpoint_pairs':meta.loc[mask,['source_before_key','source_after_key']].drop_duplicates().shape[0]})
        for contrast in c:
            fam=part[contrast].groupby('family_id').mean()
            subset_rows.append({'receiver_context':context,'subset':name,'contrast':contrast,'donor_receiver_pairs':len(part),**estimate(fam,['subset',context,name,contrast])})
            subset_families.extend({'receiver_context':context,'subset':name,'contrast':contrast,'family_id':k,'value':float(v),
                'n_pairs':int(part.xs(k,level='family_id').shape[0])} for k,v in fam.items())
subset_summary=pd.DataFrame(subset_rows);subset_coverage=pd.DataFrame(coverage);subset_family=pd.DataFrame(subset_families)
save(subset_summary,'source_prediction_subset_effects');save(subset_coverage,'source_prediction_subset_coverage');save(subset_family,'subset_family_values')
save(pair_meta.reset_index(),'all_pair_subset_membership')
donor_summary=[]
for flag in ('prediction_stable','stable_unique_top','stable_wrong','prediction_changed','source_both_correct','source_top_tie'):
    donor_summary.append({'flag':flag,'donor_pairs':int(donors[flag].sum()),'total_donor_pairs':len(donors),
        **estimate(donors.groupby('family_id')[flag].mean(),['donor_flag',flag])})
donor_summary=pd.DataFrame(donor_summary);save(donor_summary,'donor_prediction_summary')
print('Subsets are post-hoc, conditionally supported; intervals are descriptive. Stable prediction does not fix confidence.')
show(donor_summary);show(subset_coverage)
show(subset_summary[(subset_summary.receiver_context=='flip')&subset_summary.contrast.isin(['physical_vs_clean','physical_vs_random','accuracy_gain_vs_clean'])])


In [ ]:
%%cell_progress 9/13 Audit family influence and mapping boundaries
cell_stage('Audit family influence and mapping boundaries')
family_rows=[];influence=[];robust=[];boundaries=[]
for context in ('flip','stable'):
    c=contrasts(pivot_pairs(frame[frame.receiver_context==context]))
    for name,values in c.items():
        fam=values.groupby('family_id').mean();mu=float(fam.mean());leave=[]
        for k,v in fam.items():
            m=float(fam.drop(k).mean());leave.append(m)
            family_rows.append({'receiver_context':context,'contrast':name,'family_id':k,'effect':float(v)})
            influence.append({'receiver_context':context,'contrast':name,'omitted_family':k,'full_mean':mu,'leave_one_out_mean':m,'change':m-mu})
        robust.append({'receiver_context':context,'contrast':name,'positive_families':int((fam>0).sum()),'zero_families':int((fam==0).sum()),
            'negative_families':int((fam<0).sum()),'median':float(fam.median()),'min_family':float(fam.min()),'max_family':float(fam.max()),
            'leave_one_out_min':min(leave),'leave_one_out_max':max(leave),**estimate(fam,['robust',context,name])})
    joined=c.join(pair_meta[['receiver_mapping','orientation','source_mapping']])
    for boundary in ('receiver_mapping','orientation','context','source_mapping_slot','source_mapping'):
        reset=joined.reset_index()
        for value,g in reset.groupby(boundary):
            for name in c:
                fam=g.groupby('family_id')[name].mean()
                boundaries.append({'receiver_context':context,'boundary':boundary,'value':str(value),'contrast':name,
                    'donor_receiver_pairs':len(g),**estimate(fam,['boundary',context,boundary,str(value),name])})
family_effects=pd.DataFrame(family_rows);family_robustness=pd.DataFrame(robust);leave_one_out=pd.DataFrame(influence);boundary_summary=pd.DataFrame(boundaries)
save(family_effects,'family_effects');save(family_robustness,'family_robustness');save(leave_one_out,'leave_one_family_out');save(boundary_summary,'mapping_mirror_source_boundaries')
print('Family sign counts and leave-one-out means are descriptive; they do not create new independent confirmations.')
show(family_robustness[family_robustness.receiver_context=='flip'])
show(boundary_summary[(boundary_summary.receiver_context=='flip')&(boundary_summary.boundary=='receiver_mapping')])


In [ ]:
%%cell_progress 10/13 Separate dose response, answer changes and source copying
cell_stage('Separate dose response, answer changes and source copying')
direction_rows=[];dose_contrasts=[];copy_rows=[];transition_rows=[]
for context,g in frame.groupby('receiver_context'):
    for kind,part in g.groupby('kind'):
        for metric in ('margin_shift','cf_best_other_gain','cf_accuracy_gain','relative_payload_norm'):
            direction_rows.append({'receiver_context':context,'kind':kind,'metric':metric,
                **estimate(part.groupby('family_id')[metric].mean(),['direction',context,kind,metric])})
        for a,b in itertools.product((False,True),repeat=2):
            hit=(part.clean_cf_success==a)&(part.cf_success==b)
            rate=hit.groupby(part.family_id).mean()
            transition_rows.append({'receiver_context':context,'kind':kind,'clean_cf_success':a,'patch_cf_success':b,
                'records':int(hit.sum()),'total_records':len(part),**estimate(rate,['transition',context,kind,a,b])})
    w=pivot_pairs(g)
    for name,v in {'full_minus_twice_half':w.physical-2*w.physical_half,'positive_plus_negative':w.physical+w.reverse,
                   'full_minus_half':w.physical-w.physical_half,'positive_minus_negative':w.physical-w.reverse}.items():
        dose_contrasts.append({'receiver_context':context,'contrast':name,**estimate(v.groupby('family_id').mean(),['dose_contrast',context,name])})
    for kind,part in g[g.kind!='zero'].groupby('kind'):
        for code,mask,metric in [('semantic',part.source_semantic_conflict,'source_semantic_copy_gain'),
                                  ('letter',part.source_label_conflict,'source_label_copy_gain')]:
            selected=part.loc[mask]
            copy_rows.append({'receiver_context':context,'kind':kind,'copy_type':code,'eligible_records':len(selected),
                **estimate(selected.groupby('family_id')[metric].mean(),['copy_gain',context,kind,code])})
direction_summary=pd.DataFrame(direction_rows);dose_tests=pd.DataFrame(dose_contrasts);copy_gains=pd.DataFrame(copy_rows);transitions=pd.DataFrame(transition_rows)
save(direction_summary,'dose_direction_and_behavior');save(dose_tests,'dose_symmetry_descriptive_contrasts')
save(copy_gains,'source_copy_change_over_clean');save(transitions,'three_candidate_transitions')
same_as_original=float(physical[physical.receiver_context=='flip'].source_semantic.eq(physical[physical.receiver_context=='flip'].base_semantic).mean())
print('Source gold semantic equals recipient BEFORE answer, fraction in flip:',same_as_original)
print('Copy rates must be compared with clean; retaining the original answer can look like source copying.')
show(direction_summary[direction_summary.metric.isin(['margin_shift','cf_accuracy_gain'])]);show(dose_tests)
show(copy_gains[(copy_gains.receiver_context=='flip')&copy_gains.kind.isin(['physical','random','mapping'])])


In [ ]:
%%cell_progress 11/13 Measure natural-counterfactual state alignment
cell_stage('Measure natural-counterfactual state alignment')
# Only scalar geometry reconstructible from the saved traces; no hidden-state loading.
n=traces.induced_norm.to_numpy(float);m=traces.natural_cf_norm.to_numpy(float)
p=traces.projection_on_natural_cf.to_numpy(float);c=traces.cosine_to_natural_cf.to_numpy(float)
require(np.isfinite(n).all() and np.isfinite(m).all() and (n>=0).all() and (m>=0).all(),'Invalid trace norms.')
defined=m>1e-12;cos_defined=n*m>1e-12
require(np.isfinite(p[defined]).all() and np.isnan(p[~defined]).all(),'Projection undefined-mask mismatch.')
require(np.isfinite(c[cos_defined]).all() and np.isnan(c[~cos_defined]).all(),'Cosine undefined-mask mismatch.')
require((np.abs(c[cos_defined])<=1+1e-5).all(),'Cosine out of range.')
require(np.allclose(p[cos_defined],c[cos_defined]*n[cos_defined],atol=2e-5,rtol=5e-5),'Projection/cosine/norm arithmetic mismatch.')
require((np.abs(p[defined])<=n[defined]+2e-5+5e-5*n[defined]).all(),'Projection exceeds displacement norm.')
require(traces.groupby(['input_key','layer']).natural_cf_norm.nunique().eq(1).all(),'Natural endpoint norm changed across conditions.')
fraction=np.full(len(traces),np.nan);relative_distance=np.full(len(traces),np.nan)
fraction[defined]=p[defined]/m[defined]
sq=n[defined]**2+m[defined]**2-2*p[defined]*m[defined]
scale=n[defined]**2+m[defined]**2
require((sq>=-1e-4*np.maximum(scale,1.)).all(),'Impossible negative squared distance.')
relative_distance[defined]=np.sqrt(np.maximum(sq,0))/m[defined]
alignment=traces.copy();alignment['cf_aligned_fraction']=fraction
alignment['relative_distance_to_cf']=relative_distance;alignment['closeness_gain_over_clean']=1-relative_distance
extra=frame[['input_key','condition_id','context','source_mapping_slot','prediction_stable','stable_unique_top','source_both_correct','margin_shift','cf_best_other_gain','cf_accuracy_gain']]
alignment=alignment.merge(extra,on=['input_key','condition_id'],validate='many_to_one')
alignment_rows=[];alignment_coverage=[];alignment_pairs=[]
METRICS=['cosine_to_natural_cf','cf_aligned_fraction','closeness_gain_over_clean']
for (context,layer,kind),g in alignment.groupby(['receiver_context','layer','kind']):
    for metric in METRICS:
        valid=g[metric].notna();fam=g.loc[valid].groupby('family_id')[metric].mean()
        alignment_rows.append({'receiver_context':context,'layer':int(layer),'kind':kind,'metric':metric,'valid_records':int(valid.sum()),
            'total_records':len(g),**estimate(fam,['alignment',context,int(layer),kind,metric])})
for (context,layer),g in alignment[alignment.kind!='zero'].groupby(['receiver_context','layer']):
    for metric in METRICS:
        w=g.pivot(index=PAIR,columns='kind',values=metric)
        meta=pair_meta.reindex(w.index)
        for subset in ('all','stable_unique_top','source_both_correct','all_v2_pairs'):
            selected=meta[subset].fillna(False).astype(bool)
            for control in ('random','background','mapping'):
                mask=selected&w.physical.notna()&w[control].notna()
                v=(w.physical-w[control]).loc[mask];fam=v.groupby('family_id').mean()
                alignment_coverage.append({'receiver_context':context,'layer':int(layer),'subset':subset,'metric':metric,'control':control,
                    'valid_pairs':len(v),'eligible_pairs':int(selected.sum()),**estimate(fam,['alignment_contrast',context,int(layer),subset,metric,control])})
                alignment_pairs.extend({'receiver_context':context,'layer':int(layer),'subset':subset,'metric':metric,'control':control,
                    'family_id':fid,'difference':float(val)} for fid,val in fam.items())
alignment_summary=pd.DataFrame(alignment_rows);alignment_contrasts=pd.DataFrame(alignment_coverage)
correlations=[];joint=[]
for (context,layer),g in alignment[alignment.kind=='physical'].groupby(['receiver_context','layer']):
    fam=g.groupby('family_id')[['cf_aligned_fraction','closeness_gain_over_clean','margin_shift','cf_best_other_gain','cf_accuracy_gain']].mean()
    for fid,r in fam.iterrows():joint.append({'receiver_context':context,'layer':int(layer),'family_id':fid,**r.to_dict()})
    for metric in ('cf_aligned_fraction','closeness_gain_over_clean'):
        pair=fam[[metric,'margin_shift']].dropna()
        rho=(float(pair[metric].rank().corr(pair.margin_shift.rank())) if len(pair)>=3 and pair[metric].nunique()>1 and pair.margin_shift.nunique()>1 else None)
        correlations.append({'receiver_context':context,'layer':int(layer),'state_metric':metric,'n_families':len(pair),'spearman_descriptive':rho})
joint=pd.DataFrame(joint);correlations=pd.DataFrame(correlations)
save(alignment,'state_alignment_per_record');save(alignment_summary,'state_alignment_summary')
save(alignment_contrasts,'state_alignment_paired_controls');save(pd.DataFrame(alignment_pairs),'state_alignment_family_contrasts')
save(joint,'state_output_family_means');save(correlations,'state_output_descriptive_correlations')
print('Positive projection and shorter distance are separate questions. Correlations are not mediation tests.')
show(alignment_summary[(alignment_summary.receiver_context=='flip')&(alignment_summary.kind=='physical')])
show(alignment_contrasts[(alignment_contrasts.receiver_context=='flip')&(alignment_contrasts.subset=='all')&(alignment_contrasts.layer.isin([49,55]))])
show(correlations)


In [ ]:
%%cell_progress 12/13 Display family, donor and state-response figures
cell_stage('Display family, donor and state-response figures')
plt.rcParams.update({'figure.dpi':110,'savefig.dpi':170,'axes.spines.top':False,'axes.spines.right':False})
def finish(fig,name):
    if SOURCE_SYNTHETIC:fig.suptitle('SYNTHETIC ONLY — implementation validation',color='darkred',fontsize=12)
    fig.tight_layout();fig.savefig(FIGURE_DIR/(name+'.png'),bbox_inches='tight');display(fig);plt.close(fig)
def error_plot(ax,table,labels,title):
    x=np.arange(len(table));v=table.estimate.to_numpy(float);lo=table.ci_low.to_numpy(float);hi=table.ci_high.to_numpy(float)
    ax.errorbar(x,v,yerr=[np.maximum(0,v-lo),np.maximum(0,hi-v)],fmt='o',capsize=3,color='steelblue')
    ax.axhline(0,color='gray',lw=.8);ax.set_xticks(x,labels,rotation=25,ha='right');ax.set_title(title)
fig,axes=plt.subplots(1,2,figsize=(13,4.8))
sub=family_effects[(family_effects.receiver_context=='flip')&(family_effects.contrast=='physical_vs_clean')].sort_values('effect')
axes[0].bar(np.arange(len(sub)),sub.effect,color=np.where(sub.effect>=0,'steelblue','salmon'));axes[0].axhline(0,color='gray',lw=.8)
axes[0].set(xlabel='Physical family (sorted)',ylabel='Mean signed margin gain',title='Effect across all families')
s=boundary_summary[(boundary_summary.receiver_context=='flip')&(boundary_summary.boundary=='receiver_mapping')&(boundary_summary.contrast=='physical_vs_clean')].sort_values('value')
error_plot(axes[1],s,s.value.tolist(),'All six receiver mappings (descriptive CI)');finish(fig,'01_family_and_mapping')
fig,axes=plt.subplots(1,2,figsize=(13,5))
for ax,metric,title in zip(axes,['physical_vs_clean','accuracy_gain_vs_clean'],['Signed margin gain','Counterfactual accuracy gain']):
    s=subset_summary[(subset_summary.receiver_context=='flip')&(subset_summary.contrast==metric)].set_index('subset').reindex(SUBSET_NAMES).reset_index()
    labels=[name+'\nn='+str(int(n)) for name,n in zip(s['subset'],s.n_families)]
    error_plot(ax,s,labels,title);ax.tick_params(axis='x',labelsize=8)
finish(fig,'02_source_prediction_subsets')
fig,axes=plt.subplots(1,2,figsize=(12,4.6));order=['zero','physical_half','physical','reverse','random','background','mapping']
for ax,metric,title in zip(axes,['margin_shift','cf_accuracy_gain'],['Signed margin response','Three-candidate accuracy change']):
    s=direction_summary[(direction_summary.receiver_context=='flip')&(direction_summary.metric==metric)].set_index('kind').reindex(order).reset_index()
    error_plot(ax,s,order,title)
finish(fig,'03_direction_and_behavior')
fig,axes=plt.subplots(1,2,figsize=(12,4.8))
for ax,metric,title in zip(axes,['cf_aligned_fraction','closeness_gain_over_clean'],['Projection / natural CF displacement','Closeness gain over clean (positive = closer)']):
    for kind in ('physical','physical_half','reverse','random','background','mapping'):
        s=alignment_summary[(alignment_summary.receiver_context=='flip')&(alignment_summary.kind==kind)&(alignment_summary.metric==metric)].sort_values('layer')
        ax.plot(s.layer,s.estimate,marker='o',label=kind)
    ax.axhline(0,color='gray',lw=.8);ax.set_xticks([48,49,55]);ax.set_title(title);ax.set_xlabel('Decoder layer (0-based)')
axes[1].legend(fontsize=8);finish(fig,'04_state_alignment_and_distance')
fig,axes=plt.subplots(1,2,figsize=(11,4.6))
for ax,layer in zip(axes,[49,55]):
    s=joint[(joint.receiver_context=='flip')&(joint.layer==layer)]
    ax.scatter(s.cf_aligned_fraction,s.margin_shift,color='steelblue');ax.axhline(0,color='gray',lw=.8);ax.axvline(0,color='gray',lw=.8)
    ax.set(xlabel='Mean projection / natural CF displacement',ylabel='Mean signed margin gain',title=f'L{layer}: family-level association only')
finish(fig,'05_state_and_output_association')


In [ ]:
%%cell_progress 13/13 Display the Chinese report and export the CPU review bundle
cell_stage('Display the Chinese report and export the CPU review bundle')
def md_table(table):
    def fmt(v):
        if pd.isna(v):return 'NA'
        if isinstance(v,(float,np.floating)):return f'{v:.5f}'
        return str(v).replace('|','/')
    return '\n'.join(['| '+' | '.join(map(str,table.columns))+' |','| '+' | '.join(['---']*len(table.columns))+' |']+
        ['| '+' | '.join(fmt(v) for v in row)+' |' for row in table.itertuples(index=False,name=None)])
main_subsets=subset_summary[(subset_summary.receiver_context=='flip')&subset_summary.contrast.isin(['physical_vs_clean','physical_vs_random','physical_vs_background','physical_vs_mapping','accuracy_gain_vs_clean'])]
main_alignment=alignment_contrasts[(alignment_contrasts.receiver_context=='flip')&(alignment_contrasts.subset=='all')&(alignment_contrasts.layer.isin([49,55]))]
natural_accuracy=behavior.groupby('role').agg(accuracy=('correct','mean'),inputs=('correct','size')).reset_index()
source_state='**SYNTHETIC ONLY：模拟实现验证，不能作为科研结果。**' if SOURCE_SYNTHETIC else '来源：真实 Native 执行结果包；本轮只做 CPU 重算和次级审计。'
text='\n\n'.join(['# Native 物理更新：家族、来源预测与状态对齐审计',source_state,
    f'源 ZIP 哈希：`{SOURCE_HASH}`。核对 {N_FAMILIES} 个家族、{len(behavior)} 个 clean 输入、{len(frame)} 次干预和 {len(traces)} 条状态测量。',
    '原文件哈希、运行身份、逐条件记录校验和、候选分数重建和原四项主检验均通过。原主结论未被次级分析替换。',
    '## 原确认性结果',md_table(primary),'## 原准确率变化',md_table(behavior_effects),'## 自然端点能力',md_table(natural_accuracy),
    'margin 是两候选相对分数，准确率是包含 balanced 的三分类结果；分数上升不自动等于最终正确。stable 的原门槛仅是预设的下降界限。',
    '## 家族稳健性',md_table(family_robustness[family_robustness.receiver_context=='flip']),
    '正负家族数和逐一剔除范围用于定位影响；不是新的独立复现。全映射／镜像／来源边界详见 mapping_mirror_source_boundaries.csv。',
    '## 来源预测是否真的保持不变',md_table(donor_summary),'### 支持数',md_table(subset_coverage),'### Flip 配对子集效应',md_table(main_subsets),
    '来源正确答案不变与模型预测不变不同。stable_unique_top 要求两端预测相同且都无并列第一；stable_wrong 是稳定但错误的来源。子集由 clean 记录决定，各对照与物理更新用相同配对行。',
    '子集区间是事后、条件于入选样本的描述性区间；小支持不能证明效应消失，不同子集均值不直接构成因果交互。即使预测稳定，来源置信度和混合上下文信息仍可变化；相关逐来源数值已导出。',
    '## 方向、剂量与行为',md_table(dose_tests),md_table(direction_summary[direction_summary.metric=='cf_accuracy_gain']),
    'full_minus_twice_half 和 positive_plus_negative 用于描述线性／奇对称偏离，不证明线性物理算法。完整三分类转变计数见 three_candidate_transitions.csv。',
    '## 来源复制相对 clean 的变化',md_table(copy_gains[(copy_gains.receiver_context=='flip')&copy_gains.kind.isin(['physical','random','mapping'])]),
    f'Flip 中来源语义与接收端原答案相同的比例为 {same_as_original:.3f}。因此原始来源复制率混入“保留原判断”；本轮报告相对 clean 的变化，仍不将其视为已识别的复制机制。',
    '## L49/L55 对真实反事实的状态对齐',md_table(main_alignment),
    '正投影只说明沿自然反事实方向有分量；closeness_gain_over_clean 为正才表示相对距离缩短。高对齐不等于正确输出，也不证明该状态为自然中介。自然位移为零或余弦无定义时保留 NA，表中同时给出配对覆盖数。',
    '## 状态与输出的家族关联',md_table(correlations),'此处为描述性 Spearman 关联，没有将多次干预当独立样本，没有进行自然中介估计。',
    '## 技术核验与可见范围',str(quality_counts)+f'；旧标量门槛会误报的新记录数：{false_old_gate}。',
    '包内隐藏张量未提供；仅核对已经记录的 v2 向量核验摘要。旧 v1 行保留原检查，all_v2_pairs 子集要求该配对的所有非零条件均为 v2；它可能缺少部分家族，不是完整队列的替代。',
    '## 下一步取舍',
    '先看全体家族结果、稳定来源子集及支持数，再结合三候选行为和 L49/L55 对齐。若效应依赖少数家族或预测改变的来源，应收窄论文表述；不能删除这些样本后重写原主结果。',
    '若稳定来源仍有一致效应且优于控制，可增强有限范围的物理更新迁移解释；仍不能排除置信度变化。若对齐或更近没有得到支持，应分别保留行为分数效应与状态几何边界，不将两者拼成完整中介链。',
    '需要更强机制归因时，再用新冻结数据做来源预测／置信度控制或有针对性的中介干预；本轮不重新选择层、方向、剂量或家族。'])+'\n'
report=OUTPUT_ROOT/'NATIVE_PHYSICAL_UPDATE_CPU_REPORT_ZH.md';report.write_text(text,encoding='utf-8');display(Markdown(text))
key_results={'source_zip_hash':SOURCE_HASH,'source_run_id':RUN_ID,'source_synthetic':SOURCE_SYNTHETIC,'primary_reproduced':True,
    'original_decision':decision,'primary':records(primary),'accuracy_effects':records(behavior_effects),
    'family_robustness':records(family_robustness),'donor_prediction_summary':records(donor_summary),
    'subset_coverage':records(subset_coverage),'subset_effects':records(subset_summary),'alignment_contrasts':records(alignment_contrasts),
    'state_output_association':records(correlations),'recorded_audit_coverage':quality_counts,'new_confirmatory_tests':False}
write_json(OUTPUT_ROOT/'KEY_RESULTS.json',key_results)
require(file_sha(SOURCE_ZIP)==SOURCE_HASH,'Source ZIP changed during analysis.')
outputs={p.relative_to(OUTPUT_ROOT).as_posix():file_sha(p) for p in sorted(OUTPUT_ROOT.rglob('*')) if p.is_file() and p.name not in ('CPU_RUN_COMPLETE.json','cpu_review_bundle.zip','cpu_review_bundle.tmp')}
write_json(OUTPUT_ROOT/'CPU_RUN_COMPLETE.json',{'status':'complete','source_zip_hash':SOURCE_HASH,'source_run_id':RUN_ID,
    'source_synthetic':SOURCE_SYNTHETIC,'analysis_spec':ANALYSIS_SPEC,'outputs':outputs})
bundle=OUTPUT_ROOT/'cpu_review_bundle.zip';tmp=bundle.with_suffix('.tmp')
with zipfile.ZipFile(tmp,'w',zipfile.ZIP_DEFLATED) as z:
    for name in outputs:z.write(OUTPUT_ROOT/name,name)
    z.write(OUTPUT_ROOT/'CPU_RUN_COMPLETE.json','CPU_RUN_COMPLETE.json')
tmp.replace(bundle)
print('Report:',report,'\nCPU review bundle:',bundle,'\nFinished. Save and return the executed notebook with its visible tables and figures.')
